# Matrix completion via recommendation system example

This example demonstrates the use of matrix completion techniques on a recommendation system.  The recommendation system uses data from the [360K Last.fm dataset](http://ocelma.net/MusicRecommendationDataset/lastfm-360K.html).

In [3]:
!pip install -U implicit h5py

  Using cached implicit-0.7.2.tar.gz (70 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached h5py-3.16.0-cp313-cp313-win_amd64.whl.metadata (3.1 kB)
Using cached h5py-3.16.0-cp313-cp313-win_amd64.whl (3.2 MB)
Failed to build implicit


  error: subprocess-exited-with-error
  
  × Building wheel for implicit (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [410 lines of output]
      C:\Users\yashf\AppData\Local\Temp\pip-build-env-kuy7kxna\overlay\Lib\site-packages\setuptools\dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: MIT License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ********************************************************************************
      
      !!
        self._finalize_license_expression()
      
      
      --------------------------------------------------------------------------------
      -- Tryi

In [4]:
# retrieving last.fm dataset
from implicit.datasets.lastfm import get_lastfm
import numpy as np
import pandas as pd
from scipy import sparse
import os
from pathlib import Path

ModuleNotFoundError: No module named 'implicit'

## Downloading and saving the Last.fm dataset

In [5]:
filepath = r'datasets/'
Path(filepath).mkdir(exist_ok=True)

if not os.path.exists(filepath + r'artist_user_plays.npz'):
    # save our dataset in sparse format
    artists, users, artist_user_plays = get_lastfm()

    sparse.save_npz(filepath + r'artist_user_plays.npz', artist_user_plays)
    np.save(filepath + 'artists.npy', artists)
    np.save(filepath + 'users.npy', users)
else:
    # load our dataset into original format
    artist_user_plays = sparse.load_npz(filepath + r'artist_user_plays.npz')
    artists = np.load(filepath + 'artists.npy', allow_pickle=True)
    users = np.load(filepath + 'users.npy', allow_pickle=True)

NameError: name 'Path' is not defined

In [ ]:
# investigate the content of the downloaded dataset

In [ ]:
# return the dimensions of data


In [ ]:
# return the number of non-missing entries 
artist_user_plays.count_nonzero()

In [ ]:
# investigate the proportion of non-zero entries
artist_user_plays.count_nonzero()/np.prod(artist_user_plays.shape)

## Preparing the data
Okapi BM25 (Best Matching) scoring is a ranking algorithm used by search engines to estimate the relevance of items to a given search query, based on the frequency of occurrences and the size of the reference pool.  The origin of the algorithm is used in search terms in a pool of documents.

For completeness, the BM25 score of query $Q=\{q_1, \ldots, q_n\}$ for a document $D$ is calculated as:

$$\text{BM25}(D, Q) = \sum_{i=1}^{n} \frac{IDF(q_i) \cdot f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot (1 - b + b \cdot \frac{|D|}{\text{avgD}})},$$
where
- $IDF(q_i)$ is the inverse document frequency of term $q_i$.
- $f(q_i, D)$ is the term frequency of $q_i$ in the document $D$.
- $k_1$ and $b$ are parameters controlling term saturation and document length normalization.
- $D$ is the length of the document.
- $\text{avgD}$ is the average document length in the corpus.


In [ ]:
from implicit.nearest_neighbours import bm25_weight

# using the weighting function for normalization
artist_user_plays = bm25_weight(artist_user_plays, K1=100, B=0.8)
user_plays = artist_user_plays.T.tocsr()


## Training the model with alternating least squares

In [ ]:
from implicit.als import AlternatingLeastSquares

# using alternating least squares algorithm
model = AlternatingLeastSquares(factors=16, regularization=0.05, alpha=2.0)
model.fit(user_plays)

## Similar artists recommendation

In [ ]:
# generate similar artist recommendation
list(artists).index('Beyonce')
artist_id = 45721
ids, scores = model.similar_items(artist_id)
pd.DataFrame({'artist': artists[ids], 'score': scores})

## User-specific recommendation

In [ ]:
# generate user-based recommendation